# Session 4 · de-code agentic coding

*Democratizing Coding with AI — Thu 8 Oct*

<nils.holmberg@iko.lu.se>

## Topics

<!-- table of contents: update when a section heading is renamed or moved -->
- Recap: session 3
    - From idea to spec
- 1 Safety first!
    - 1.1 From chat to agents
    - 1.2 Common agent tools
    - 1.3 AGENTS.md: rules the agent reads first
    - 1.4 Guardrails and slash commands
    - 1.5 Models and tokens
- 2 Specs and agentic AI
    - 2.1 Agentic coding tools
    - 2.2 Create a project
    - 2.3 The spec loop
    - 2.4 Iteration 1: build from a finished spec
    - 2.5 Iteration 2: change the spec
    - 2.6 Iteration 3: an app that saves data
    - 2.7 Iteration 4: a notebook app with packages
    - 2.8 Your turn
- 3 Agent basics
    - 3.1 Brain, loop and tools
    - 3.2 Tools are ordinary functions
    - 3.3 The loop in 30 lines
    - 3.4 Agents at work: one agent, then a team
    - 3.5 Design your own agent, and going further
- [Slides](https://de-code-ai.netlify.app/coding/skill-school/sessions/04-agentic-coding-slides.html)
- [Notebook](https://colab.research.google.com/github/nils-holmberg/de-code/blob/main/jnb/skill-school/04-agentic-coding.ipynb)

In [ ]:
#| echo: false
import matplotlib.pyplot as plt

iterations = list(range(1, 8))
tasks_done = [1, 3, 4, 7, 9, 12, 14]

fig, ax = plt.subplots()
ax.step(iterations, tasks_done, where="mid", color="#2f9e44", linewidth=2)
ax.plot(iterations, tasks_done, "o", color="#2f9e44", markersize=6)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e6e6e6", linewidth=0.8)
ax.set_axisbelow(True)
ax.set_xlabel("agent iteration")
ax.set_ylabel("tasks completed")
ax.set_title("Illustrative: spec → build → check, iteration by iteration")
plt.tight_layout()
plt.show()

# Recap: session 3

Yesterday we described apps in plain language, turned the descriptions into specs, and let
Gemini write the code. Today an AI agent takes over that work — and we look at how agents
work inside.

## From idea to spec

The specs and apps from the course are collected in the GitHub repository
[cca-cce/osm-edu](https://github.com/cca-cce/osm-edu). The word-cloud app started as an idea
in two sentences:

In [ ]:
#| eval: false
idea_prompt = """
Make a single-page web app where I can paste a text and click Submit to see its most
important words as a word cloud. Clean the text first: lowercase it, split it into
words, reduce each word to its base form, and leave out common words like "the" and "and".
"""

Gemini turned that idea into a working page:
[Word Cloud Generator](https://cca-cce.github.io/osm-edu/gemini-code-1791366306868.html).
From the same idea we then wrote a fuller spec, `NLP-WEB-SPEC.md`, which pins down the
layout, the libraries, the processing steps and the acceptance checks. In section 2.4 an
agent builds the app from that spec.

# 1 Safety first!

## 1.1 From chat to agents

A language model such as Gemini works by **predicting the next word**, over and over, based
on everything written so far. In a chat, that produces an answer — but only text: you copy
the code, run it, and paste back the result.

An **AI agent** adds two things:

1. **Tools** — functions the model can call to act outside the chat: read and write files,
   search a folder, run a command in a terminal, look something up on the web.
2. **A loop** — the agent plans a step, calls a tool, reads the result, and decides the next
   step, until the task is done or it needs your input.

That makes agents powerful: you describe the goal, and the agent does the work in a folder
on *your* computer. It also makes them risky: a tool that can write a file can overwrite
one, and a tool that can run a command can delete things. So we start with safety.

## 1.2 Common agent tools

Most coding agents — Antigravity included — offer a similar set of tools. The last column
is a rough guide to how much damage a mistake could do.

| Tool | What it does | Example | Risk |
|---|---|---|---|
| Read file | Reads the contents of a file | read `NLP-WEB-SPEC.md` | low |
| List / find files | Lists folders or finds files by name | find all `*.csv` files | low |
| Search in files | Finds text inside files | where is `DV_INDEX` used? | low |
| Create / write file | Creates a new file or replaces one | write `app.py` | medium — can overwrite |
| Edit file | Changes part of an existing file | rename a variable in `app.py` | medium |
| Run terminal command | Runs a command in a shell | `quarto render report.qmd` | high — can do anything you can |
| Web search | Searches the internet | "Gradio slider example" | low — but results can be wrong |
| Fetch web page | Downloads a page or file | read a library's documentation | low–medium |
| Ask the user | Pauses and asks you a question | "Which column is the outcome?" | none — this is a good sign |
| Plan / to-do list | Writes and updates its own task list | 1. load data 2. describe … | none |

Reading is safe; **writing, editing and running commands** are where you want guardrails.

## 1.3 AGENTS.md: rules the agent reads first

An `AGENTS.md` file is a plain Markdown file of **standing instructions**. The agent reads it
at the start of every conversation, before your prompt, so the rules apply without you
repeating them. Put one in a project folder for that project; Antigravity also reads global
rules from `~/.gemini/AGENTS.md` (or `GEMINI.md`) for all your projects.

Here is a real example, used while building this course. It stores the text; save it as
`AGENTS.md` in your project folder.

In [ ]:
#| eval: false
agents_md = """
## Local Python Environment

This repository uses the Python virtual environment:
~/lib/python/uv-osm

---

### Environment Activation

When running any Python-related command (python, pytest, quarto render with python, etc.),
ALWAYS prefix the command with:

source ~/lib/python/uv-osm/bin/activate && <command>

If multiple commands are needed, prefer a single bash -lc invocation that activates once and runs all steps.

---

### Package Management Policy (IMPORTANT)

This environment uses **uv** for dependency management.

DO NOT install, upgrade, or remove any Python packages without explicitly asking first.

If a dependency is required:

1. Explain why the package is needed.
2. Confirm with the user.
3. Install it using:

   source ~/lib/python/uv-osm/bin/activate && uv pip install <package>

Rules:

- Prefer `uv pip install` over `pip install`.
- Never use plain `pip install`.
- Never install packages globally.
- Never modify requirements files or pyproject.toml without confirmation.
- If a package is missing, check first before proposing installation.

---

### File System Safety Policy (IMPORTANT)

You may read and inspect files anywhere.

You may create, modify, or delete files ONLY inside:

    /tmp/
    ~/tmp/

This includes any subdirectory under /tmp/ or ~/tmp/ (e.g., /tmp/project/, /tmp/test/output.txt, ~/tmp/project/).

You must NEVER modify files outside /tmp/ or ~/tmp/ without explicit confirmation.

If changes to project files are required:
1. Write the proposed version to /tmp/ or ~/tmp/.
2. Show or explain the changes.
3. Ask for approval before modifying the original file.

Treat everything outside /tmp/ and ~/tmp/ as read-only unless explicitly approved.
"""

with open("AGENTS.md", "w") as f:
    f.write(agents_md)

The file has three policies, each closing a typical risk:

- **Environment:** always use one specific Python environment, so the agent does not run
  code with the wrong setup.
- **Packages:** never install or remove software without asking first — installs change
  your computer beyond the project.
- **File system:** read anything, but write only in scratch folders; changes to real files
  are proposed first and need your approval.

Note the style: short, direct, with the important parts marked **IMPORTANT** and concrete
examples. Write rules the way you would brief a new colleague.

## 1.4 Guardrails and slash commands

Rules are one layer of safety. Combine them with these habits:

- **Approve actions.** By default the agent asks before writing files or running commands.
  Keep it that way; never start it with `--dangerously-skip-permissions`.
- **Plan first** with `/plan` (or `agy --mode plan`): the agent only investigates and proposes
  until you approve. **Use the sandbox** (`agy --sandbox`) to limit what commands can reach.
- **Work in a copy** of your data, **keep checkpoints** (git, or `/rewind` to go back and
  `/diff` to see what changed), and keep **secrets** out of the project folder.

In [ ]:
#| eval: false
# Start the CLI agent in a project folder, in plan mode, inside the sandbox
!cd ~/projects/wordcloud-app && agy --mode plan --sandbox

Commands that start with `/` control the agent itself instead of being sent as a prompt.
Ten useful ones:

| Command | What it does |
|---|---|
| `/help` | Lists commands and keyboard shortcuts |
| `/plan` | Plan mode: investigate and propose, change nothing yet |
| `/grill-me` | The agent interviews you about your idea, one question at a time |
| `/model` | Shows or switches the model (`/model <name>`) |
| `/effort` | Sets how hard the model thinks (low to max) |
| `/context` | Shows what fills the context window |
| `/usage` | Shows token use and remaining quota |
| `/diff` | Shows the changes the agent made |
| `/rewind` | Goes back to an earlier point, undoing changes |
| `/resume` | Continues an earlier conversation |

**`/grill-me`** is especially useful before building an app: instead of guessing what you
mean, the agent asks question after question — who uses the app, which data, what counts as
done — until the idea is clear. Combine it with plan mode, and end by asking for a spec:

In [ ]:
#| eval: false
grill_prompt = """
/plan /grill-me I want a small app that shows word clouds and sentence sentiment for
the text files in this folder. When you have no more questions, write the result as
APP-SPEC.md with purpose, data, features, constraints and acceptance checks.
"""

## 1.5 Models and tokens

An agent can run on different models. The trade-off is always speed and cost against
capability:

| Model family | Good for | Note |
|---|---|---|
| Gemini Flash (low / medium / high) | everyday tasks, quick edits, most course work | fast and cheap; start here |
| Gemini Pro | harder problems, bigger plans, tricky bugs | slower, uses more quota |
| Claude (Sonnet, Opus) | a second opinion, long careful reasoning | also available in Antigravity |
| GPT-OSS | an open-weight alternative | useful for comparison |

The **effort** level (`low` to `max`) sets how long the model thinks before acting. Switch
with `/model` and `/effort`, ask another model once with `/model <name> <prompt>`, and list
what your account can use with `agy models`.

Models read and write **tokens** — pieces of words, roughly ¾ of a word each. Everything the
agent sees counts: your prompt, `AGENTS.md`, every file it reads, every command output and its
own replies. All of it must fit in the **context window**, the model's working memory; when it
fills up, the agent *compacts* older parts into a summary and details can get lost.

- Check usage with `/context`, `/usage` and `/quota`.
- Point the agent to the files it needs instead of letting it read everything.
- Keep `AGENTS.md` and specs short, and start a new conversation for a new task.

# 2 Specs and agentic AI

## 2.1 Agentic coding tools

Every large AI maker now offers a coding agent: a model with tools and a loop that works on
the files in a project folder (section 1.1). They differ in models, prices and interfaces,
but they work the same way — and they all follow a good spec and an AGENTS.md:

| Maker | Agent | Where it runs |
|---|---|---|
| Google | [Antigravity](https://antigravity.google/), [Gemini CLI](https://github.com/google-gemini/gemini-cli) | desktop app, code editor, terminal |
| Anthropic | [Claude Code](https://claude.com/product/claude-code) | terminal, code editors, desktop and web |
| OpenAI | [Codex](https://openai.com/codex/) | terminal, code editors, cloud |
| GitHub (Microsoft) | [Copilot](https://github.com/features/copilot) agent mode | VS Code and GitHub |
| Anysphere | [Cursor](https://cursor.com) | its own code editor |

In this course we use **Google Antigravity**, which works with a Google account. It puts the
same agent behind three interfaces, all downloaded from
[antigravity.google](https://antigravity.google/download).

| Product | What you see | Suits | Start it with |
|---|---|---|---|
| Antigravity app | A desktop app for managing agents, plans and results | beginners; working without reading code | the app icon |
| Antigravity IDE | A code editor with the agent built in | reviewing code and edits closely | the app icon, or `antigravity-ide` |
| Antigravity CLI | The agent in a terminal window | quick tasks, scripts, remote computers | `agy` |

- The **app** is an *agent manager*: you describe the task and follow the agent's plan, its
  steps and its results (*artifacts*) without opening a code editor — the most
  beginner-friendly start.
- The **IDE** is a code editor in the style of Visual Studio Code, with the agent in a side
  panel. You see every edit in context and can accept or reject it line by line.
- The **CLI**, `agy`, runs the same agent in a terminal: quick to start, works on remote
  computers over SSH, and runs single prompts from scripts.

In [ ]:
#| eval: false
# Interactive session in the current folder
!agy

# Safer starts: plan mode, sandboxed terminal, a chosen model and effort
!agy --mode plan --sandbox --model gemini-3.1-pro-high --effort high

# One prompt, no conversation: prints the answer and exits
!agy -p "Summarise NLP-WEB-SPEC.md in three bullet points"

# Continue the most recent conversation, list available models, update the CLI
!agy -c
!agy models
!agy update

The commands are written with `!` so they could run from a notebook; in a normal terminal,
leave the `!` out. In Colab they do not work — the CLI runs on your own computer.

## 2.2 Create a project

Agents work in a **project folder** (the *workspace*). Everything the agent may touch lives
there, so make a fresh one for each app. Every iteration below uses the same layout:

```
my-app/
├── AGENTS.md          rules: environment, packages, where it may write
└── <APP>-SPEC.md      the task: what to build and how to check it
```

1. Create an empty folder named after the app, for example `wordcloud-app`.
2. Save `AGENTS.md` (section 1.3) and the app's spec in it.
3. Open the folder in the Antigravity app or IDE and confirm that you trust it — or open a
   terminal in the folder and run `agy`.

The agent reads both files before it starts, so your prompts can stay short.

## 2.3 The spec loop

A useful app spec has the same parts as in session 3, kept short: **purpose**, **data**,
**features**, **constraints** and **acceptance checks**. With an agent, a spec is not written
once — it is the thing you keep improving:

1. Write or update the spec.
2. Ask the agent for a plan (`/plan`) and read it.
3. Approve the plan; the agent builds.
4. Check the result against the acceptance checks — the agent first, then you.
5. If something is wrong or missing, **change the spec, not the code**, and go back to step 2.

We go round this loop four times, with one new thing to learn each time:

| Iteration | App | Runs in | New this time |
|---|---|---|---|
| 1 | Word cloud | the browser, one file | an agent builds and tests from a finished spec |
| 2 | Word cloud, extended | the browser, one file | changing a spec and reviewing the change |
| 3 | Course notes | the browser, three files | a bigger spec, and data that is saved |
| 4 | Edge detector | a Jupyter notebook | packages to install; an interface inside the notebook |

The specs are stored in cells; run them in Colab to save each one as a file for the agent.

## 2.4 Iteration 1: build from a finished spec

In session 3 you gave this spec to Gemini and copied its code into a file yourself. Now an
agent does the whole job, from the same spec:

| | Gemini chat (session 3) | Antigravity agent (session 4) |
|---|---|---|
| Writes `wordcloud.html` | you copy the code into a file | the agent creates the file |
| Opens and tests it | you | the agent, against the acceptance checks |
| Fixes errors | you paste the error back into the chat | the agent reads the error and edits the file |
| Your role | copy, paste, check, report back | approve the plan, review the result |

This is the same specification as in session 3. If you saved `NLP-WEB-SPEC.md` then, copy it
into the project folder; otherwise run the cell in Colab, download the file, and put it there.

In [ ]:
#| eval: false
web_spec = """
# SPEC: Word cloud (single-page web app)

## 1. Purpose
Paste any English text and see its most important words as a word cloud,
in the browser, without Python.

## 2. Files and libraries
- Exactly one file, wordcloud.html, with the CSS and JavaScript inside it.
  No build step, no server.
- Load the libraries as ES modules from the jsdelivr CDN, with pinned major versions:
  wink-nlp@2 with wink-eng-lite-web-model@1 (NLP), d3@7 and d3-cloud@1 (word cloud).
- Must work when wordcloud.html is opened directly from the folder (a file:// address)
  in current Chrome, Firefox, Edge and Safari. An internet connection is needed for
  the libraries.

## 3. Layout
- Wide screens (700 px and wider): two columns. Left: a labelled text area and a
  Submit button. Right: the word cloud.
- Narrow screens (below 700 px): one column, text area and button on top, word cloud
  below.
- The text area starts with a short sample paragraph, so Submit works right away.

## 4. Processing (when Submit is clicked)
1. Read: take the text from the text area; if it is empty, show a short message
   instead of a cloud.
2. Clean: lowercase; remove web addresses, numbers and extra whitespace.
3. Tokenise with wink-nlp (readDoc).
4. Lemmatise: replace each word with its lemma (its.lemma), e.g. "running" -> "run".
5. Filter: keep words of three or more letters (a-z); drop stop words
   (its.stopWordFlag) and punctuation.
6. Count how often each lemma occurs and keep the 100 most frequent.

## 5. Word cloud
- Layout with d3-cloud, drawn as SVG; font size scaled to frequency, the most frequent
  lemma largest; words horizontal.
- Hovering a word shows the lemma and its count.
- A line under the cloud: number of tokens and number of distinct lemmas.
- Clicking Submit again replaces the old cloud.

## 6. Acceptance checks
- A1 Opening wordcloud.html from the folder shows the page with no errors in the
  browser console.
- A2 Paste a text, click Submit: a word cloud appears within a few seconds.
- A3 The cloud contains no stop words such as "the", "and" or "was".
- A4 The verbs "runs", "ran" and "running" all count as one lemma, "run".
- A5 At 400 px width the cloud is below the text area; on a desktop it is to the right.
- A6 Submitting a new text replaces the cloud; an empty text shows the message.
"""

with open("NLP-WEB-SPEC.md", "w") as f:
    f.write(web_spec)

Then give the agent a short prompt — the details are in the spec. Start in plan mode, read
the plan, and approve it:

In [ ]:
#| eval: false
run_prompt = """
/plan Read AGENTS.md and NLP-WEB-SPEC.md. Build wordcloud.html as specified.
Open it in a browser, test every acceptance check (A1-A6, including a 400 px wide
window) and report which pass. Fix anything that fails. If you cannot open a browser,
tell me exactly what to check by hand.
"""

# The same as a single command in the CLI
!agy --mode plan -p "Read AGENTS.md and NLP-WEB-SPEC.md and build the app."

Finally, compare the two versions. Open the agent's `wordcloud.html` next to the one you made
with Gemini in session 3, paste the same text into both, and look at the clouds. Do they
show the same words? If not, ask the agent why — different stop words or lemmas are the
usual reasons. And check A4 yourself: "runs", "ran" and "running" should all count as "run".

Here is a version an Antigravity agent built from this spec, published as a web page:
[word-cloud app](https://cca-cce.github.io/osm-edu/apps/wordcloud.html). For more sample app
specs, see the [apps folder](https://github.com/cca-cce/osm-edu/tree/main/apps) of the
course's GitHub repository.

## 2.5 Iteration 2: change the spec

Users always want one more thing. Here the word cloud gets colour and a download button. In
the spec loop, the change goes into the **spec first**: the spec stays the single, up-to-date
description of the app, and the agent then brings the code in line with it.

In [ ]:
#| eval: false
spec_update = """
## 5. Word cloud (changed)
- Colour each word by its frequency, on a sequential colour scale from light (rare)
  to dark (frequent); every colour must stay readable on a white background.
- A "Download PNG" button saves the current cloud as wordcloud.png.

## 6. Acceptance checks (added)
- A7 The most frequent lemma has the darkest colour, and the downloaded PNG shows the
  same cloud as the page.
"""

Ask the agent to change the spec, and to show you the change before it touches the code:

In [ ]:
#| eval: false
update_prompt = """
/plan Update NLP-WEB-SPEC.md: replace section 5 with the new version below and add
check A7. Show me the changed lines of the spec first. After I approve, update
wordcloud.html to match the spec and re-check A1-A7.
"""

Review what changed with `/diff`. If the new version is worse, `/rewind` takes the project
back to before the change (section 1.4) — then improve the wording of the spec and try again.

## 2.6 Iteration 3: an app that saves data

Not every app needs a server. A web browser can run an app made of three plain files —
HTML for the structure, CSS for the look, and JavaScript for the behaviour. You open
`index.html` by double-clicking it, and the app runs on your own computer: no install, no
account, no internet connection. There is no data file to load; instead the app *creates*
data — your notes — and stores it in the browser.

What the parts of this spec do:

- **Files and constraints** make the app *local*: no frameworks to install, nothing loaded
  from the internet, and it must work when opened straight from a folder.
- **Data model and storage**: a data model says exactly what one note *is*; storage says
  where notes are kept — here `localStorage`, a small database that every browser has for
  each page.
- **Look and accessibility**: saying how the app should look and who must be able to use it
  avoids an app that only works for its builder.

In [ ]:
#| eval: false
notes_spec = """
# SPEC: Course notes (local web app)

## 1. Purpose and users

A simple note-taking app for course participants to write down ideas, prompts and
results during the sessions. One user, one computer, no account.

## 2. Files and constraints

- Exactly three files in one folder: index.html, style.css, app.js.
- Plain HTML, CSS and JavaScript. No frameworks, no libraries, no CDN links, no build
  step, no server.
- Must work when index.html is opened directly from the folder (a file:// address),
  also without an internet connection, in current Chrome, Firefox, Edge and Safari.

## 3. Data model and storage

- A note has: id (unique text), title (text), body (text), tags (list of words),
  created and updated (date and time).
- All notes are saved in the browser's localStorage under the key "course-notes" as
  JSON, after every change.
- Export: a button that downloads all notes as course-notes-<date>.json.
- Import: a button that reads such a file and adds its notes, skipping ids that already
  exist.

## 4. Features and layout

- N1 Two columns: a list of notes on the left, the selected note in an editor on the
  right. On narrow screens (below 700 px), show one column at a time.
- N2 "New note" button; the new note opens in the editor with the title selected.
- N3 Edit title, body and tags (comma-separated); changes save automatically.
- N4 Delete a note after a confirmation question.
- N5 Search box that filters the list by title and body as you type.
- N6 Click a tag to show only notes with that tag; click again to show all.
- N7 The list is sorted by last updated, newest first, and shows title, date and the
  first line of the body.
- N8 Keyboard shortcuts: Ctrl/Cmd+N new note, Ctrl/Cmd+F focus search, Esc clears the
  search.

## 5. Look and accessibility

- Clean, readable design: system font, comfortable line length, enough contrast.
- Follow the computer's light or dark mode (prefers-color-scheme).
- Every control has a visible label or an accessible name; everything works with the
  keyboard, with a visible focus outline.

## 6. Acceptance checks

- A1 Opening index.html from the folder shows the app with no errors in the browser
  console.
- A2 A new note with a title, body and two tags is still there after closing and
  reopening the browser.
- A3 Searching for a word in a body shows only notes containing that word.
- A4 Export, delete all notes, import the exported file: every note is back.
- A5 The app works with the computer offline.
- A6 At 400 px width, all features are reachable.

## 7. How to run

Write this into a short README.md:
1. Keep index.html, style.css and app.js together in one folder.
2. Double-click index.html, or drag it into a browser window.
3. Notes are stored in that browser on that computer only. Use Export regularly as a
   backup, and Import to move notes to another browser or computer.
"""

with open("NOTES-SPEC.md", "w") as f:
    f.write(notes_spec)

In [ ]:
#| eval: false
notes_prompt = """
Read AGENTS.md and NOTES-SPEC.md. Plan first, then build the three files, open
index.html in a browser, and check every acceptance criterion.
"""

## 2.7 Iteration 4: a notebook app with packages

The last app runs inside a **Jupyter notebook**, so it works in Colab and in the Antigravity
IDE alike. It uses [Gradio](https://www.gradio.app/) (session 3), which shows a small web
interface — uploads, sliders, images — directly below the cell with `demo.launch()`. It also
needs packages that may not be installed yet, and this is where the package rule in
AGENTS.md matters: the agent must ask before it installs anything.

Computer vision apps work on images. **Edge detection** finds the outlines in a picture —
where brightness changes sharply — and is a classic first step before recognising shapes or
objects. This spec uses [OpenCV](https://opencv.org/) and the Canny edge detector, with two
sliders so you can see how its thresholds change the result.

In [ ]:
#| eval: false
cv_spec = """
# SPEC: Edge detector

## 1. Purpose
Show how edge detection works: upload a picture, adjust two thresholds, and see the
edges appear.

## 2. Input
- An image uploaded by the user (JPG or PNG).
- If none is uploaded, a sample image drawn in code (shapes and text on a gradient),
  so the app works without any files.

## 3. Processing (OpenCV)
1. Convert to grayscale.
2. Smooth with a Gaussian blur (kernel size from a slider: 1, 3, 5 or 7).
3. Canny edge detection with a low and a high threshold (sliders, 0-255).
4. Optionally draw the edges in colour on top of the original image.

## 4. Interface (Gradio, shown in the notebook)
- Image upload, three sliders (blur, low threshold, high threshold), an overlay checkbox.
- The original and the edge image side by side, updated when a slider moves.
- The share of edge pixels shown as a percentage.
- A button to download the edge image as PNG.

## 5. Constraints
- One notebook cell: %pip install opencv-python-headless gradio
- Use opencv-python-headless (no desktop windows) so it runs in Colab.
- Run with demo.launch() so the interface appears below the cell.

## 6. Acceptance checks
- A1 Without an upload, the sample image and its edges are shown.
- A2 Raising the low threshold gives fewer edge pixels.
- A3 If the low threshold is above the high one, the app explains the problem instead
  of failing.
- A4 The downloaded PNG matches the edge image shown.
"""

with open("CV-APP-SPEC.md", "w") as f:
    f.write(cv_spec)

In [ ]:
#| eval: false
cv_prompt = """
Read AGENTS.md and CV-APP-SPEC.md. Plan first, then write the app as one Jupyter
notebook cell in edge_detector.ipynb, run it, and check every acceptance criterion.
"""

Here is the code an agent wrote from this spec, from the notebook
[`apps/edge_detector.ipynb`](https://github.com/cca-cce/osm-edu/blob/main/apps/edge_detector.ipynb)
in the course's GitHub repository. The first cell installs Gradio; the second builds the
interface and launches it below the cell. Run both in Colab, or open the
[notebook in Colab](https://colab.research.google.com/github/cca-cce/osm-edu/blob/main/apps/edge_detector.ipynb)
directly. With `share=True`, Gradio also prints a temporary public link to the app.

In [ ]:
#| eval: false
!pip install -q gradio

In [ ]:
#| eval: false
import os
import tempfile

import cv2
import gradio as gr
import numpy as np


def make_sample(width=720, height=480):
    # A colourful gradient makes soft and strong edges easy to compare.
    x = np.linspace(0, 1, width, dtype=np.float32)[None, :]
    y = np.linspace(0, 1, height, dtype=np.float32)[:, None]
    image = np.empty((height, width, 3), dtype=np.uint8)
    image[..., 0] = np.clip(35 + 145 * x + 35 * y, 0, 255)
    image[..., 1] = np.clip(35 + 95 * (1 - x) + 90 * y, 0, 255)
    image[..., 2] = np.clip(110 + 100 * (1 - y) + 20 * x, 0, 255)

    cv2.rectangle(image, (55, 65), (285, 250), (245, 245, 250), -1)
    cv2.rectangle(image, (55, 65), (285, 250), (25, 35, 55), 7)
    cv2.line(image, (55, 160), (250, 160), (210, 210, 210), 3, cv2.LINE_AA)
    cv2.circle(image, (505, 155), 92, (245, 185, 45), -1, cv2.LINE_AA)
    cv2.circle(image, (505, 155), 92, (30, 40, 65), 7, cv2.LINE_AA)
    cv2.line(image, (65, 365), (650, 310), (250, 250, 250), 14, cv2.LINE_AA)
    cv2.putText(
        image, "EDGE", (225, 420), cv2.FONT_HERSHEY_DUPLEX,
        2.1, (22, 28, 45), 5, cv2.LINE_AA
    )
    return image


SAMPLE_IMAGE = make_sample()
OUTPUT_PATH = os.path.join(tempfile.gettempdir(), "edge_detector_result.png")


def as_rgb_uint8(image):
    if image is None:
        return SAMPLE_IMAGE.copy()

    image = np.asarray(image)
    if image.dtype != np.uint8:
        image = np.clip(image, 0, 255).astype(np.uint8)
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    elif image.shape[2] == 4:
        image = cv2.cvtColor(image, cv2.COLOR_RGBA2RGB)
    return image


def save_png(rgb_image):
    # Save the same array returned to the preview, so the PNG matches it exactly.
    cv2.imwrite(OUTPUT_PATH, cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR))
    return OUTPUT_PATH


def detect_edges(image, blur_size, low_threshold, high_threshold, overlay):
    original = as_rgb_uint8(image)
    blur_size = int(blur_size)
    low_threshold = int(low_threshold)
    high_threshold = int(high_threshold)

    if low_threshold > high_threshold:
        empty = np.zeros_like(original)
        return (
            original,
            empty,
            "⚠️ Low threshold must be less than or equal to the high threshold.",
            save_png(empty),
        )

    gray = cv2.cvtColor(original, cv2.COLOR_RGB2GRAY)
    smoothed = gray if blur_size == 1 else cv2.GaussianBlur(gray, (blur_size, blur_size), 0)
    edges = cv2.Canny(smoothed, low_threshold, high_threshold)
    edge_mask = edges > 0

    if overlay:
        result = original.copy()
        accent = np.array([255, 70, 45], dtype=np.float32)
        result[edge_mask] = (
            0.25 * result[edge_mask].astype(np.float32) + 0.75 * accent
        ).astype(np.uint8)
    else:
        result = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)

    count = int(np.count_nonzero(edge_mask))
    share = 100.0 * count / edge_mask.size
    status = f"**Edge pixels:** {share:.2f}% · {count:,} of {edge_mask.size:,} pixels"
    return original, result, status, save_png(result)


initial_original, initial_edges, initial_status, initial_download = detect_edges(
    None, 3, 60, 140, False
)

theme = gr.themes.Soft(
    primary_hue="orange",
    secondary_hue="slate",
    neutral_hue="slate",
)

css = """
.gradio-container { max-width: 1180px !important; }
.app-title { margin-bottom: 0.1rem !important; }
.app-note { color: var(--body-text-color-subdued); margin-top: 0 !important; }
.status-card { padding: 0.8rem 1rem; border: 1px solid var(--border-color-primary); border-radius: 12px; }
"""

with gr.Blocks(title="Edge detector") as demo:
    gr.Markdown("# Edge detector", elem_classes=["app-title"])
    gr.Markdown(
        "Upload a JPG or PNG, then tune the blur and Canny thresholds. "
        "With no upload, the built-in sample is used.",
        elem_classes=["app-note"],
    )

    with gr.Row():
        with gr.Column(scale=1, min_width=280):
            image_input = gr.Image(
                label="Upload image",
                sources=["upload"],
                type="numpy",
            )
            blur = gr.Slider(
                minimum=1, maximum=7, value=3, step=2, label="Gaussian blur kernel"
            )
            low = gr.Slider(0, 255, value=60, step=1, label="Low threshold")
            high = gr.Slider(0, 255, value=140, step=1, label="High threshold")
            overlay = gr.Checkbox(value=False, label="Overlay coloured edges")
            stats = gr.Markdown(initial_status, elem_classes=["status-card"])
            download = gr.DownloadButton(
                "Download edge image (PNG)", value=initial_download, variant="primary"
            )

        with gr.Column(scale=2, min_width=420):
            with gr.Row():
                original_output = gr.Image(
                    value=initial_original, label="Original", interactive=False
                )
                edge_output = gr.Image(
                    value=initial_edges, label="Edges", interactive=False
                )

    inputs = [image_input, blur, low, high, overlay]
    outputs = [original_output, edge_output, stats, download]
    image_input.change(detect_edges, inputs=inputs, outputs=outputs)
    for control in [blur, low, high, overlay]:
        control.input(detect_edges, inputs=inputs, outputs=outputs)

demo.queue().launch(theme=theme, css=css, share=True, debug=False)

Compare the code with the spec: can you find where each acceptance check (A1-A4) is
handled? `make_sample` covers A1, the threshold check in `detect_edges` covers A3, and
`save_png` saves exactly the image that is shown, for A4.

## 2.8 Your turn

Now go round the whole loop with an idea of your own:

1. Let the agent interview you in plan mode: `/plan /grill-me I want an app that …`
   (section 1.4). Answer its questions until it can write the spec.
2. Ask it to save the spec as `SPEC.md`. Read it, and correct anything it got wrong.
3. Let the agent build the app and check every acceptance check.
4. Change one feature in the spec — for example a Markdown preview for the notes, or a
   second edge detector (Sobel) next to Canny — add an acceptance check for it, and run the
   loop again.

# 3 Agent basics

## 3.1 Brain, loop and tools

So far we have used a finished agent. How does one work inside? Section 1.1 gave the short
answer: an agent is a **model**, some **tools**, and a **loop**. In this part we build a tiny
agent in Python to see each piece. It runs on your own computer with
[Ollama](https://ollama.com) and a local model: no account, no API key, and nothing leaves
your machine.

*This part is adapted from Alexandros Sopasakis' workshop "Build the research colleague you
wish you had" (Agentic AI: Research and Applications, Lund University, 28 September 2026).*

| Part | What it is | Who does what |
|---|---|---|
| Brain | a language model, here running locally in Ollama | **chooses**: answer now, or call a tool |
| Tools | ordinary Python functions | **act**: read a file, run code, save a result |
| Loop | a few lines of Python around the model | **runs** the chosen tool and reports back |

The useful idea: once the loop is written, a new agent is just **a name, a job description,
and a list of tools** it may use.

## 3.2 Tools are ordinary functions

A tool is a normal Python function. Its **name, type hints and docstring** are what the model
sees: they tell it what the tool does and which inputs it needs — the same idea as the tools
of an MCP server in session 5. Safety is built into the code, not just asked for in a prompt:
every path goes through `safe_path`, so the agent cannot read anything outside its workspace
folder (compare the guardrails in section 1.4).

In [ ]:
#| eval: false
from pathlib import Path
import csv

WORKSPACE = Path("workspace").resolve()   # the only folder the agent may use


def safe_path(relative_path: str) -> Path:
    """Resolve a path inside the workspace; refuse anything outside it."""
    path = (WORKSPACE / relative_path).resolve()
    if path != WORKSPACE and WORKSPACE not in path.parents:
        raise ValueError("Path must stay inside the workspace")
    return path


def read_text_file(path: str) -> str:
    """Read a UTF-8 text file from the workspace.

    Args:
        path: File relative to the workspace, e.g. 'abstract.txt'.
    """
    return safe_path(path).read_text(encoding="utf-8")


def inspect_csv(path: str) -> str:
    """Read a small CSV file from the workspace and return its columns and rows.

    Args:
        path: CSV file relative to the workspace, e.g. 'measurements.csv'.
    """
    with safe_path(path).open(newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    return f"columns={list(rows[0]) if rows else []}\nrows={rows}"

## 3.3 The loop in 30 lines

The `Agent` class holds the loop, so you write it once. Its system prompt is the agent's name,
its job and a few working rules. Then it repeats: send the conversation and the tool list to
the model; if the model calls a tool, Python runs it and adds the result to the conversation;
if the model answers without a tool, that is the final answer. `max_steps` stops an agent
that goes round in circles.

In [ ]:
#| eval: false
from dataclasses import dataclass, field
from typing import Callable
from ollama import chat


@dataclass
class Agent:
    name: str
    role: str                                  # the job description
    tools: list[Callable] = field(default_factory=list)
    model: str = "qwen3:8b"                    # any local Ollama model with tool calling
    max_steps: int = 8                         # safety limit for the loop

    def run(self, goal: str) -> str:
        system = (f"You are {self.name}.\n\nJOB\n{self.role}\n\nWORKING RULES\n"
                  "- Use a tool whenever the answer depends on what it can provide.\n"
                  "- Never invent a tool result.\n"
                  "- Stop when the goal is met, or say clearly what blocks you.")
        messages = [{"role": "system", "content": system},
                    {"role": "user", "content": goal}]
        available = {fn.__name__: fn for fn in self.tools}

        for step in range(self.max_steps):
            reply = chat(model=self.model, messages=messages, tools=self.tools or None).message
            messages.append(reply)
            if not reply.tool_calls:                   # no tool needed: this is the answer
                print("MODEL REPLIED ← here is the answer")
                return reply.content
            for call in reply.tool_calls:              # the model chose; Python executes
                name, args = call.function.name, dict(call.function.arguments or {})
                print(f"MODEL REPLIED ← call {name}({args})")
                fn = available.get(name)
                try:
                    result = fn(**args) if fn else f"ERROR: no tool called {name}"
                except Exception as exc:               # errors go back to the model
                    result = f"ERROR: {type(exc).__name__}: {exc}"
                print(f"    OBSERVATION ← {str(result)[:300]}")
                messages.append({"role": "tool", "tool_name": name, "content": str(result)})
        return f"Stopped after {self.max_steps} steps."

The lines starting with `MODEL REPLIED ←` show what came back from the model *before* Python
did anything with it. The choice of tool is always the model's; Python only checks that the
tool exists and runs it.

## 3.4 Agents at work: one agent, then a team

The first agent is a *Data Detective*. Its workspace holds a small file of eight
measurements, one of which looks wrong:

In [ ]:
#| eval: false
Path("workspace").mkdir(exist_ok=True)
Path("workspace/measurements.csv").write_text(
    "id,measurement\n1,10.1\n2,10.3\n3,9.9\n4,10.2\n5,10.0\n6,25.0\n7,10.1\n8,9.8\n"
)

data_detective = Agent(
    name="Data Detective",
    role="Answer questions about the files in the workspace. Inspect data or read text "
         "as the question requires, and explain the evidence you used.",
    tools=[inspect_csv, read_text_file],
)

data_detective.run("Inspect measurements.csv. Identify the suspicious measurement "
                   "and compute the mean of the remaining measurements.")

The model decides by itself to open the file. The output looks something like this:

```text
MODEL REPLIED ← call inspect_csv({'path': 'measurements.csv'})
    OBSERVATION ← columns=['id', 'measurement'] rows=[{'id': '1', 'measurement': '10.1'}, ...
MODEL REPLIED ← here is the answer
Measurement 6 (25.0) is far from the others (9.8 to 10.3) and looks like an error.
Without it, the mean of the remaining seven measurements is 10.06.
```

Check the answer yourself: the seven other values average 10.057. Given a question about a
text file instead, the same agent picks the other tool — nothing in the code changes.

Agents can also use **other agents as tools**. Wrap each specialist in a function, and a
*Chief* agent gets the specialists as its tools and decides who works next. Each specialist
has a smaller job and fewer tools, which makes its work easier to check:

In [ ]:
#| eval: false
def as_tool(agent: Agent) -> Callable[[str], str]:
    """Wrap an agent as a tool, so another agent can delegate work to it."""
    def delegate(task: str) -> str:
        return agent.run(task)
    delegate.__name__ = agent.name.lower().replace(" ", "_")
    delegate.__doc__ = (f"Delegate a task to {agent.name}. Role: {agent.role}\n\n"
                        "Args:\n    task: The specific task for the specialist.")
    return delegate


Path("workspace/abstract.txt").write_text(
    "Synthetic teaching text. In an online experiment (n = 100), a message about the "
    "benefits of exercise raised intentions to exercise more than a control message "
    "(Cohen's d = 0.58). We conclude that such messages change behaviour in all groups."
)

analyst = Agent("Paper Analyst",
                "Read the material and state the main claim, the evidence and the assumptions.",
                [read_text_file])
reviewer = Agent("Reviewer 2",
                 "Act as a sceptical reviewer: challenge overclaims and say what the "
                 "evidence does not support.",
                 [read_text_file])
chief = Agent("Chief Scientist",
              "Coordinate the goal: decide which specialist works next, give them a clear "
              "task, combine their results, and finish with a short evidence report. "
              "Never invent results.",
              [as_tool(analyst), as_tool(reviewer)], max_steps=12)

chief.run("Is the conclusion in abstract.txt supported by its evidence?")

A good team run ends with the reviewer's point: the study measured *intentions*, in one
sample, so "change behaviour in all groups" goes beyond the evidence.

## 3.5 Design your own agent, and going further

Designing an agent is a short list of questions about the task, not about code:

In [ ]:
#| eval: false
agent_design = """
1. GOAL      What outcome should the agent achieve?
2. TOOLS     What must it be able to do: read files, run code, search, save?
3. EVIDENCE  What must it show before you trust the result?
4. STOP      When must it stop, or ask you for approval?
5. TEAM      (optional) Is one agent enough, or would separate roles be more reliable?
"""

my_agent = Agent(
    name="Methods Editor",
    role="Check whether a methods description is precise enough to repeat the study.",
    tools=[read_text_file],
)

The answers become the agent's three lines: a name, a job description and its tools. A job
description is a small spec — the same habit as in part 2.

To run the code in this part on your own computer, install [Ollama](https://ollama.com),
download a model that supports tool calling, and install the Python package (ask first if an
AGENTS.md says so):

In [ ]:
#| eval: false
!ollama pull qwen3:8b      # about 5 GB; larger models follow instructions better
!pip install ollama

Agent frameworks package the same loop with memory, streaming and deployment. The free
[LangChain Academy](https://github.com/langchain-ai/langchain-academy) course has notebooks
that open in Colab, for example a
[tool-calling agent](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/agent.ipynb)
and the same
[agent with memory](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/agent-memory.ipynb).
They use OpenAI models by default, which need a paid API key; with small changes they also
work with Gemini.